<a href="https://colab.research.google.com/github/Lucaaa31/Anomaly-Segmentation/blob/master/notebooks/Step7_AnomalyDetection_Erfnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 7

## Link with the drive and pull of the repo


In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Anomaly-Segmentation
#!git pull origin master

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation


## Install the dependencies

In [2]:
!pip install -r requirements.txt

## Libraries imported

In [3]:
import os
import cv2
import glob
import torch
import torch.nn.functional as F
import random
from PIL import Image
import numpy as np
from models.erfnet.erfnet import ERFNet
import wget
import torch.nn as nn
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from pathlib import Path

seed_everything(42, verbose=False)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

## Evaluation script
Here we have our pipeline to test the model using three post-hoc methods:
- MaxLogit
- MSP
- Max Entropy

The datasets that we will use are three:
- RoadAnomaly
- LostAndFound
- fs_static

### Load My State Dict
Load the weights of a pre-trained model, it is useful because it handles even the fact that PyTorch saves the models with ".module" as prefix.

In [4]:
def load_my_state_dict(model, state_dict):
        own_state = model.state_dict()
        for name, param in state_dict.items():
            if name not in own_state:
                if name.startswith("module."):
                    own_state[name.split("module.")[-1]].copy_(param)
                else:
                    print(name, " not loaded")
                    continue
            else:
                own_state[name].copy_(param)
        return model

### Preprocessing
Pipeline for resizing the input and the groud_truth images.

In [5]:
# Resize the input images in 512x1024 and convert them in a tensor
input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

# Resize the ground truth masks in the same dimensions
target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)


### Configuration

Change the parameters in order to adapt it to your local dataset:

*   **Input**: Where are the images to analyze
*   **Model Paths**: Path to the model (the .py script for load it) and his weigths (.pth)
*   **Weights**: Name of the file that contains the weight of the model
*   **Script**: Name of the script that load our model
*   **Subset**: Used for specify if we have to work on the validation or training set
*   **Data Directory**: Path to the root of the dataset (it has to be where there are the 2 subfolders images and labels_masks)
*   **Workers**: How many parallels process can be executed (more --> more RAM)
*   **Batch Size**: How many images it analyzes
*   **Hardware**: Use CPU instead of GPU
*   **Method**: Choose of the methods

In [43]:

dataset_to_use = "RoadAnomaly21" # @param ["fs_static", "RoadAnomaly21", "RoadAnomaly", "FS_LostFound_full", "RoadObsticle21"]
subset = "val"
methods = "MSP" # @param ["MSP", "max_logit", "max_entropy"]



input_pattern = f"/content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/{dataset_to_use}/images/*"
datadir = f"/content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/{dataset_to_use}/"
loadDir = "/content/drive/MyDrive/Anomaly-Segmentation/models/erfnet/"
loadWeights = "erfnet_pretrained.pth"
loadModel = "erfnet.py"


num_workers = 4
batch_size = 1
device = "cuda" if torch.cuda.is_available() else "cpu"


NUM_CHANNELS = 3
NUM_CLASSES = 20

class Args:
    def __init__(self):
        self.input = input_pattern
        self.loadDir = loadDir
        self.loadWeights = loadWeights
        self.loadModel = loadModel
        self.subset = subset
        self.datadir = datadir
        self.num_workers = num_workers
        self.batch_size = batch_size
        self.cpu = device
        self.method = methods

args = Args()
print(f"Selected parameters: {vars(args)}")

Selected parameters: {'input': '/content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/*', 'loadDir': '/content/drive/MyDrive/Anomaly-Segmentation/models/erfnet/', 'loadWeights': 'erfnet_pretrained.pth', 'loadModel': 'erfnet.py', 'subset': 'val', 'datadir': '/content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/', 'num_workers': 4, 'batch_size': 1, 'cpu': 'cuda', 'method': 'MSP'}


## Post-hoc methods


In [37]:
def get_softmax(logits):
    probs = F.softmax(logits, 1)

    msp_score = 1.0 - np.max(probs.squeeze(0).data.cpu().numpy(), axis=0)
    return msp_score

def get_entropy(logits):
    probs = get_softmax(logits)
    num_classes = probs.shape[0]
    entropy = torch.div(
        torch.sum(-probs * torch.log(probs + 1e-7), dim=1),
        torch.log(torch.tensor(num_classes).float())
    )
    return entropy.squeeze(0).data.cpu().numpy().astype("float32")

def get_maxlogit(logits):
    return 1.0 - np.max(logits.squeeze(0).data.cpu().numpy(), axis=0)


In [44]:
anomaly_score_list = []
ood_gts_list = []

# Initialize model and weigth paths
modelpath = args.loadDir + args.loadModel
weightspath = args.loadDir + args.loadWeights

print ("Loading model: " + modelpath)
print ("Loading weights: " + weightspath)

# Initialization of the ERFNet
model = ERFNet(NUM_CLASSES)

if (args.cpu):
    model = torch.nn.DataParallel(model).cuda()


model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
print ("Model and weights LOADED successfully")
model.eval()


for path in glob.glob(os.path.expanduser(str(args.input))):
    print(f"Processing image: {path.strip('/')}")
    images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
    # images = images.permute(0,3,1,2) --> removed because it does not work
    with torch.no_grad():
        result = model(images)


    match args.method:
        case "MSP":
            anomaly_result = get_softmax(result)
        case "max_logit":
            anomaly_result = get_maxlogit(result)
        case "max_entropy":
            anomaly_result = get_entropy(result)


    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
        ood_gts = np.where((ood_gts==2), 1, ood_gts)
    if "FS_LostFound_full" in pathGT:
        ood_gts = np.where((ood_gts==0), 255, ood_gts)
        ood_gts = np.where((ood_gts==1), 0, ood_gts)
        ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts==14), 255, ood_gts)
        ood_gts = np.where((ood_gts<20), 0, ood_gts)
        ood_gts = np.where((ood_gts==255), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue
    else:
            ood_gts_list.append(ood_gts)
            anomaly_score_list.append(anomaly_result)
    del result, anomaly_result, ood_gts, mask
    torch.cuda.empty_cache()



Loading model: /content/drive/MyDrive/Anomaly-Segmentation/models/erfnet/erfnet.py
Loading weights: /content/drive/MyDrive/Anomaly-Segmentation/models/erfnet/erfnet_pretrained.pth
Model and weights LOADED successfully
Processing image: content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/0.png
Processing image: content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/1.png
Processing image: content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/2.png
Processing image: content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/3.png
Processing image: content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/4.png
Processing image: content/drive/MyDrive/Anomaly-Segmentation/dataset/Anomaly_Validation_Dataset/RoadAnomaly21/images/5.png
Processing image: content/drive/MyDrive/Anom

In [46]:
# Create/append the file for save the results
if not os.path.exists('results/Step7.txt'):
    open('results/Step7.txt', 'w').close()
file = open('results/Step7.txt', 'a')

file.write( "\n")

ood_gts = np.array(ood_gts_list)
anomaly_scores = np.array(anomaly_score_list)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

ood_out = anomaly_scores[ood_mask]
ind_out = anomaly_scores[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

print(val_label)
print(val_out)

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score: {prc_auc*100.0}')
print(f'FPR@TPR95: {fpr*100.0}')

header = f"Method: {args.method} | Dataset: {dataset_to_use}\n"
metrics = f"AUPRC: {prc_auc*100.0:.5f}% | FPR@TPR95: {fpr*100.0:.5f}%\n"
divider = "="*50 + "\n"

file.write(header + metrics + divider)
file.close()

[0. 0. 0. ... 1. 1. 1.]
[0.13432133 0.10615903 0.04751551 ... 0.48818487 0.3464849  0.6697981 ]
AUPRC score: 29.095121365386866
FPR@TPR95: 62.54877304263575
